# Init Lakehouse

## Import Helper Functions

In [1]:
from src.config_loader import load_config
from src.spark_sql_magic import sql

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


## Load Configs

In [2]:
cfg = load_config()

JOB_NAMES = cfg["spark_jobs"]["jobs"]
CATALOG = cfg["general"]["catalog"]
RAW_FOLDER = cfg["raw"]["raw_folder"]
BRONZE_NAMESPACE = cfg["general"]["namespaces"]["bronze"]
SILVER_NAMESPACE = cfg["general"]["namespaces"]["silver"]
GOLD_NAMESPACE = cfg["general"]["namespaces"]["gold"]
MONITORING_NAMESPACE = cfg["general"]["namespaces"]["monitoring"]

## Import Libraries and Start Session

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window as W
import pyspark
import datetime
import json

spark = (
    SparkSession.builder
        .appName(JOB_NAMES["init_lakehouse"])
        .getOrCreate()
)

## Select Catalog

In [4]:
spark.sql(f"USE {CATALOG};")

DataFrame[]

## Create Lakehouse Namespace

In [5]:
NAMESPACES = [
    RAW_FOLDER,
    BRONZE_NAMESPACE,
    SILVER_NAMESPACE, 
    GOLD_NAMESPACE,
    MONITORING_NAMESPACE
]

for ns in NAMESPACES:
    spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {ns};")

In [6]:
%%sql
SHOW NAMESPACES;

+----------+
|namespace |
+----------+
|silver    |
|bronze    |
|raw       |
|monitoring|
|gold      |
+----------+



In [7]:
spark.catalog.clearCache()  # clears all cached tables
spark.stop()  